# Discrete Vapor Cavity Model (DVCM) Showcase

This notebook demonstrates the physical and numerical behavior of the **Discrete Vapor Cavity Model (DVCM)** in RTHYM-MOC. It compares DVCM against the **Legacy Clamp** model using a severe valve-closure transient that triggers water-column separation and subsequent cavity collapse.

## Why use DVCM?
- **Legacy Clamp**: Clamps the hydraulic grade line (HGL) at the vapor pressure floor. It prevents non-physical negative pressures but does *not* track the physical size of the vapor pocket, and it completely misses the severe **secondary water hammer** overpressures that occur when the column separation collapses.
- **DVCM**: Solves the physical regime-switching equations for column separation. It tracks the growth and collapse of the vapor pocket volume and resolves the high-intensity secondary pressure spike generated when the separated liquid columns collide.

## 1. Setup and Dependencies

Import the required libraries. RTHYM-MOC is a high-performance C++ solver exposed to Python via pybind11.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rthym_moc as m

print(f"rthym_moc version: {getattr(m, '__version__', 'unknown')}")

## 2. Define the Pipeline Geometry and Simulation Scenarios

We set up a simple pipeline consisting of:
- An upstream reservoir `R1` (constant HGL = 150 ft)
- A long main pipe `P1` (length = 3000 ft, diameter = 12", wave speed = 4000 ft/s)
- A transient valve `V1`
- A short discharge pipe `P2` (length = 40 ft, diameter = 12", wave speed = 4000 ft/s)
- A downstream reservoir `R2` (constant HGL = 147.9 ft)

The initial flow through the system is **500 GPM** (initial velocity $V_0 \approx 1.42$ ft/s). At $t = 0.09$ seconds, the valve begins a rapid linear closure, slamming completely shut by $t = 0.10$ seconds.

In [ ]:
def build_network():
    solver = m.MOCSolver()
    
    # 1. Define Nodes
    r1 = m.NodeInput()
    r1.id = "R1"
    r1.type = "PressureBoundary"
    r1.elevation = 0.0
    r1.head = 150.0
    
    v1 = m.NodeInput()
    v1.id = "V1"
    v1.type = "Valve"
    v1.elevation = 0.0
    v1.diameter = 12.0
    v1.current_setting = 100.0
    v1.head = 148.0
    
    r2 = m.NodeInput()
    r2.id = "R2"
    r2.type = "PressureBoundary"
    r2.elevation = 0.0
    r2.head = 147.9
    
    solver.add_node(r1)
    solver.add_node(v1)
    solver.add_node(r2)
    
    # 2. Define Pipes
    p1 = m.PipeInput()
    p1.id = "P1"
    p1.from_node = "R1"
    p1.to_node = "V1"
    p1.length = 3000.0
    p1.diameter = 12.0
    p1.roughness = 130.0
    p1.flow_gpm = 500.0
    
    p2 = m.PipeInput()
    p2.id = "P2"
    p2.from_node = "V1"
    p2.to_node = "R2"
    p2.length = 40.0
    p2.diameter = 12.0
    p2.roughness = 130.0
    p2.flow_gpm = 500.0
    
    solver.add_pipe(p1)
    solver.add_pipe(p2)
    
    # 3. Define Valve Closure Schedule
    solver.set_valve_schedule(
        "V1",
        [
            (0.0, 100.0),
            (0.09, 100.0),
            (0.10, 0.0),
            (10.0, 0.0)
        ]
    )
    
    return solver

## 3. Run Simulation 1: Legacy Clamp Mode

In this run, we use the default `LegacyClamp` cavitation model. The legacy clamp is highly stable under coarse timesteps, so we run it with a standard timestep $dt = 0.01$ seconds.

In [ ]:
solver_legacy = build_network()
res_legacy = solver_legacy.run(
    total_time=10.0,
    dt=0.01,
    p_vapor_psi=-14.0,  # ~full vacuum
    cavitation_model=m.CavitationModel.LegacyClamp
)

time_legacy = np.array(res_legacy["time"])
head_legacy = np.array(res_legacy["node_head"]["V1"])
cav_legacy = np.array(res_legacy["node_cavitation"]["V1"])

print("--- Legacy Clamp Results ---")
print(f"Total Steps Simulated: {len(time_legacy)}")
print(f"Maximum HGL Head at Valve: {np.max(head_legacy):.2f} ft")
print(f"Minimum HGL Head at Valve: {np.min(head_legacy):.2f} ft")
print(f"Cavitating Steps: {np.sum(cav_legacy)}")

## 4. Run Simulation 2: DVCM Mode

Now we opt-in to the `DVCM` model.

> **Important:** Because DVCM tracks physical cavity volumes that can change rapidly during collapse, we must reduce the timestep to prevent numerical volume integration overshoot. We use **$dt = 0.0001$ seconds** to ensure numerical stability.

In [ ]:
solver_dvcm = build_network()
res_dvcm = solver_dvcm.run(
    total_time=10.0,
    dt=0.0001,
    p_vapor_psi=-14.0,  # ~full vacuum
    cavitation_model=m.CavitationModel.DVCM
)

time_dvcm = np.array(res_dvcm["time"])
head_dvcm = np.array(res_dvcm["node_head"]["V1"])
cav_active_dvcm = np.array(res_dvcm["node_cavity_active"]["V1"])
cav_volume_dvcm = np.array(res_dvcm["node_cavity_volume"]["V1"])
collapse_counts_dvcm = np.array(res_dvcm["node_cavity_collapse_count"]["V1"])

print("--- DVCM Results ---")
print(f"Total Steps Simulated: {len(time_dvcm)}")
print(f"Maximum HGL Head at Valve: {np.max(head_dvcm):.2f} ft")
print(f"Minimum HGL Head at Valve: {np.min(head_dvcm):.2f} ft")
print(f"Active Cavitation Steps: {np.sum(cav_active_dvcm)}")
print(f"Max Cavity Volume: {np.max(cav_volume_dvcm):.4f} ft³")
print(f"Total Cavity Collapses: {collapse_counts_dvcm[-1]}")

## 5. Visual Comparisons and Interpretation

Let's plot the transient head profiles at the valve to visually compare the difference between the two models.

In [ ]:
plt.figure(figsize=(12, 6))

# Plot Legacy Clamp
plt.plot(time_legacy, head_legacy, label="Legacy Clamp (dt=0.01s)", color="gray", alpha=0.7, linestyle="--")

# Plot DVCM
plt.plot(time_dvcm, head_dvcm, label="DVCM (dt=0.0001s)", color="crimson", linewidth=1.5)

plt.title("Valve HGL Head comparison: Legacy Clamp vs. DVCM", fontsize=14, fontweight="bold")
plt.xlabel("Time (s)", fontsize=12)
plt.ylabel("HGL Head (ft)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.xlim(0, 10.0)

# Highlight key events
plt.axvline(x=0.10, color="blue", linestyle=":", alpha=0.5)
plt.text(0.015, 0.6, "Valve Closed", color="blue", fontsize=10, transform=plt.gca().transAxes)

plt.tight_layout()
plt.show()

### Analysis of the Head Comparison
1. **Initial Water Hammer**: Immediately after valve closure ($t = 0.1$ s), both models calculate the initial Joukowsky pressure rise of around $590$ ft of head.
2. **Reflected Negative Wave**: At $t \approx 1.6$ seconds, the negative wave reflected from reservoir `R1` returns to the valve, causing the pressure to drop.
3. **Regime Difference**:
   - In **Legacy Clamp**, the pressure drops to the vapor pressure floor ($-32.3$ ft) and simply clamps there. After a few seconds, the pressure recovers back above the floor smoothly.
   - In **DVCM**, a vapor pocket forms. The pressure remains clamped at the vapor pressure floor while the cavity is active, but at $t \approx 5.6$ seconds, the cavity completely collapses. The water columns collide, creating a massive secondary pressure spike that peaks at **over 3,400 ft of head**! This secondary spike is more than **5 times larger** than the initial closure spike.

## 6. Cavity Volume Growth and Collapse

Let's inspect the growth and collapse of the physical vapor cavity volume at the valve as calculated by DVCM.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

# Plot cavity volume
color = 'darkcyan'
ax1.set_xlabel('Time (s)', fontsize=12)
ax1.set_ylabel('Cavity Volume (ft³)', color=color, fontsize=12)
ax1.plot(time_dvcm, cav_volume_dvcm, color=color, linewidth=2, label="Cavity Volume")
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)

# Instantiate a second axes that shares the same x-axis
ax2 = ax1.twinx()  
color = 'darkorange'
ax2.set_ylabel('Collapse Count', color=color, fontsize=12)
ax2.plot(time_dvcm, collapse_counts_dvcm, color=color, linewidth=1.5, linestyle="--", label="Collapse Count")
ax2.tick_params(axis='y', labelcolor=color)

plt.title("Vapor Cavity Volume & Collapse Count at Valve (DVCM)", fontsize=14, fontweight="bold")
fig.tight_layout()  
plt.show()

### Interpretation of Cavity Dynamics
- **Initiation**: The vapor pocket begins growing at $t \approx 1.6$ s, when the HGL head drops to the vapor limit.
- **Growth Phase**: The cavity volume grows to a peak size of approximately **110 ft³** as the upstream water column moves away from the valve.
- **Collapse Phase**: When the flow reversals push the water columns back together, the cavity volume shrinks. At $t \approx 5.6$ s, the volume reaches exactly zero.
- **Spike & Count**: At the moment of collapse, the `node_cavity_collapse_count` increments from 0 to 1, and the water-column collision generates the massive head spike observed in the previous plot.

## 7. Numerical Stability and Timestep Guidance

The DVCM model is physically realistic but is highly sensitive to the timestep size $dt$.
- **Legacy Clamp** has no state integration, making it stable under coarse grids (e.g. $dt = 0.01$ s).
- **DVCM** requires tracking a small volume over time:
  $$V_c(t) = \int (Q_{\text{out}} - Q_{\text{in}}) \, dt$$
  If $dt$ is too large, the integrated volume can overshoot or chatter, resulting in numerical instabilities (causing `NaN`/`Inf` propagation or negative volumes).
- **Recommendation**: Always use **$dt \le 0.001$ s** when `DVCM` is active (recommend **$0.0001$ s** or smaller for severe column separation events).

---

## 8. Launching in Binder

This repository is configured for Binder, which makes it easy to run and explore this notebook in your web browser without installing anything locally.

To launch this notebook in Binder:
1. Go to [mybinder.org](https://mybinder.org).
2. Enter the URL of this repository: `https://github.com/jlillywh/RTHYM-MOC`.
3. Select the branch `chore/dvcm-phase3-validation-harness`.
4. Click **Launch**.
5. Open `examples/dvcm_showcase.ipynb` and run all cells!